In [ ]:
# Import general utilities
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import os
# Import TDA utilities
import gudhi
import gudhi.representations
from ripser import Rips
from persim.landscapes import PersLandscapeExact
from persim.landscapes import plot_landscape_simple
from persim.landscapes import PersistenceLandscaper
from persim import plot_diagrams
from tadasets import torus, sphere

# Import Scikit-Learn tools
from sklearn.decomposition import PCA
from sklearn import svm
from sklearn import metrics
from sklearn.model_selection import train_test_split
from mpl_toolkits.mplot3d import Axes3D # Required for 3D plots

def savePic( filename ):
    # Ensure the output directory exists
    output_dir = os.path.join(os.getcwd(), 'Picture')
    os.makedirs(output_dir, exist_ok=True)
    # Save to the "Picture" folder
    filepath = os.path.join(output_dir, filename)
    # print(filepath)
    plt.savefig(filepath, dpi=300, bbox_inches='tight')

In [ ]:
import pandas as pd
# from google.colab import drive
# drive.mount('/content/drive')

epha2a = pd.read_csv("/workspaces/Cellular-TDA/data/Old Data/centroids Epha2a hom_table.csv")
control = pd.read_csv("/workspaces/Cellular-TDA/data/Old Data/centroids control_table.csv")


x_epha2a = epha2a['Center of the object_0']
y_epha2a = epha2a['Center of the object_1']
z_epha2a = epha2a['Center of the object_2']

x_control = control['Center of the object_0']
y_control = control['Center of the object_1']
z_control = control['Center of the object_2']

epha2_data = np.column_stack((x_epha2a, y_epha2a, z_epha2a))
control_data = np.column_stack((x_control, y_control, z_control))

In [ ]:
dgm_epha2_data   = Rips(maxdim=2).fit_transform(epha2_data)
dgm_control_data = Rips(maxdim=2).fit_transform(control_data)


fig = plt.figure(figsize=(10,10))
# Plot Data

#3D plots
ax0 = fig.add_subplot(2, 2, 1, projection='3d')
ax1 = fig.add_subplot(2, 2, 2, projection='3d')

ax1.scatter(epha2_data[:, 0], epha2_data[:, 1], epha2_data[:,2])
ax1.set_title("Epha2a")
ax0.scatter(control_data[:, 0], control_data[:, 1], control_data[:,2])
ax0.set_title("Control")
# Change rotation: 30 degrees elevation, 45 degrees azimuth
ax0.view_init(elev=45, azim=90, roll=0)
ax1.view_init(elev=45, azim=45, roll=0)
# Plot Persistence Diagrams
ax2 = fig.add_subplot(2, 2, 3)
ax3 = fig.add_subplot(2, 2, 4)

plot_diagrams(dgm_control_data, title="Control", ax=ax2)
plot_diagrams(dgm_epha2_data, title="Epha2a", ax=ax3)

savePic("cloudPoints.png")
pl_epha2 = PersLandscapeExact(dgm_epha2_data, hom_deg=1)
pl_control = PersLandscapeExact(dgm_control_data, hom_deg=1)

fig, axs = plt.subplots(1, 2)
fig.set_size_inches(12, 8)

plot_landscape_simple(pl_epha2, ax=axs[1])
axs[1].set_title("Persistence Landscape for Epha2a")
plot_landscape_simple(pl_control, ax=axs[0])
axs[0].set_title("Persistence Landscape for Control")

axs[0].get_legend().remove()
axs[1].get_legend().remove()
savePic("plusLandscape.png")

In [ ]:


# Create one figure for all 6 plots
fig = plt.figure(figsize=(18, 10))

# --- ROW 1: CONTROL GROUP (Row 1, Col 1, 2, 3) ---

# Control 3D Plot
ax1 = fig.add_subplot(2, 3, 1, projection='3d')
ax1.scatter(control_data[:, 0], control_data[:, 1], control_data[:, 2])
ax1.set_title("Control: 3D Data")
ax1.view_init(elev=45, azim=90)

# Control Persistence Diagram
ax2 = fig.add_subplot(2, 3, 2)
plot_diagrams(dgm_control_data, title="Control: Persistence Diagram", ax=ax2)

# Control Persistence Landscape
ax3 = fig.add_subplot(2, 3, 3)
plot_landscape_simple(pl_control, ax=ax3)
ax3.set_title("Control: Persistence Landscape")
if ax3.get_legend(): ax3.get_legend().remove()


# --- ROW 2: EPHA2A GROUP (Row 2, Col 1, 2, 3) ---

# Epha2a 3D Plot
ax4 = fig.add_subplot(2, 3, 4, projection='3d')
ax4.scatter(epha2_data[:, 0], epha2_data[:, 1], epha2_data[:, 2])
ax4.set_title("Epha2a: 3D Data")
ax4.view_init(elev=45, azim=45)

# Epha2a Persistence Diagram
ax5 = fig.add_subplot(2, 3, 5)
plot_diagrams(dgm_epha2_data, title="Epha2a: Persistence Diagram", ax=ax5)

# Epha2a Persistence Landscape
ax6 = fig.add_subplot(2, 3, 6)
plot_landscape_simple(pl_epha2, ax=ax6)
ax6.set_title("Epha2a: Persistence Landscape")
if ax6.get_legend(): ax6.get_legend().remove()

plt.tight_layout()
savePic("grouped_analysis.png")
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import gudhi
import gudhi.representations

# --- Data Loading and Standardization ---
treatment = pd.read_csv("/workspaces/Cellular-TDA/data/Old Data/centroids Epha2a hom_table.csv")
control = pd.read_csv("/workspaces/Cellular-TDA/data/Old Data/centroids control_table.csv")

# Extract coordinates
coords = ['Center of the object_0', 'Center of the object_1', 'Center of the object_2']
treatment_raw = treatment[coords].values
control_raw = control[coords].values

# Standardize the data before applying GUDHI
scaler = StandardScaler()
treatment_data = scaler.fit_transform(treatment_raw)
control_data = scaler.fit_transform(control_raw)

def get_gudhi_persistence(data):
    # No max_edge_length provided; GUDHI defaults to infinity
    rips = gudhi.RipsComplex(points=data, max_edge_length=1.5)
    simplex_tree = rips.create_simplex_tree(max_dimension=2)
    return simplex_tree

treatment_data_simplex_tree = get_gudhi_persistence(treatment_data)
control_data_simplex_tree = get_gudhi_persistence(control_data)

fig = plt.figure(figsize=(18, 10), dpi=150)

def set_bold_title(ax, title):
    ax.set_title(title, fontweight='bold', fontsize=18)

# --- 1. 3D Plots (Standardized Data) ---
ax0 = fig.add_subplot(2, 3, 1, projection='3d')
ax1 = fig.add_subplot(2, 3, 4, projection='3d')

ax0.scatter(control_data[:, 0], control_data[:, 1], control_data[:, 2])
ax0.view_init(elev=25, azim=45)
set_bold_title(ax0, "Control")

ax1.scatter(treatment_data[:, 0], treatment_data[:, 1], treatment_data[:, 2])
ax1.view_init(elev=25, azim=45)
set_bold_title(ax1, "Treatment")

# --- 2. Persistence Diagrams (No Gray Block) ---
ax2 = fig.add_subplot(2, 3, 2)
ax3 = fig.add_subplot(2, 3, 5)

for ax, dgm, label in zip([ax2, ax3], [control_data_simplex_tree.persistence(), treatment_data_simplex_tree.persistence()], ["Control", "Treatment"]):
    gudhi.plot_persistence_diagram(dgm, axes=ax, legend=True)
    # Remove the gray diagonal shading (gray block)
    leg = ax.get_legend()
    if leg:
        for text in leg.get_texts():
            text.set_fontsize(18)
    for poly in ax.findobj(plt.Polygon):
        poly.set_visible(False)
    set_bold_title(ax, f"{label}: Persistence Diagram ")

# --- 3. Persistence Landscapes (Default Resolution, Independent Axes) ---
ax4 = fig.add_subplot(2, 3, 3)
ax5 = fig.add_subplot(2, 3, 6)

num_layers = 5
resolution = 100 # Default GUDHI resolution
LS = gudhi.representations.Landscape(num_landscapes=num_layers, resolution=resolution)

# Filter for H1 points (dimension 1)
h1_control = control_data_simplex_tree.persistence_intervals_in_dimension(1)
h1_treatment = treatment_data_simplex_tree.persistence_intervals_in_dimension(1)

for ax, h1_data, label in zip([ax4, ax5], [h1_control, h1_treatment], ["Control", "Treatment"]):
    if len(h1_data) > 0:
        lsc = LS.fit_transform([h1_data]).reshape(num_layers, -1)
        # Independent x-axis based on birth/death range
        x_min, x_max = h1_data[:, 0].min(), h1_data[:, 1].max()
        x_vals = np.linspace(x_min, x_max, resolution)
        for i in range(num_layers):
            ax.plot(x_vals, lsc[i], label=f'Layer {i+1}')
        ax.legend(fontsize=16)
    set_bold_title(ax, f"{label}: H1 Landscape")

fig.tight_layout()
for axes in [ax2, ax3]:
    axes.set_xlim(-0.2, 1.2)
    axes.set_ylim(-0.2, 1.5)
for axes in [ax4, ax5]:
     axes.set_xlim(0.1, 1.3)
     # axes.set_ylim(-0.2, 1.5)
ax2.axhline(np.max(h1_control)*1.1, color="black")
ax3.axhline(np.max(h1_treatment)*1.1, color="black")
x_values = np.linspace(-0.2, 1.5)
ax2.plot(x_values, x_values, color="black")
ax3.plot(x_values, x_values, color="black")
savePic("ApplicationInfoRow.png")
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import gudhi
import gudhi.representations
import ipywidgets as widgets
from IPython.display import display

# --- 1. Load and Prep Data ---
treatment = pd.read_csv("/workspaces/Cellular-TDA/data/Old Data/centroids Epha2a hom_table.csv")
control = pd.read_csv("/workspaces/Cellular-TDA/data/Old Data/centroids control_table.csv")
coords = ['Center of the object_0', 'Center of the object_1', 'Center of the object_2']

scaler = StandardScaler()
treatment_data = scaler.fit_transform(treatment[coords].values)
control_data = scaler.fit_transform(control[coords].values)

# --- 2. Define the Plotting Function ---
def update_plots(max_edge_length):
    # This function runs every time the slider moves
    fig = plt.figure(figsize=(18, 10))
    plt.subplots_adjust(hspace=0.3)

    # Calculate GUDHI TDA
    def process_data(data):
        rips = gudhi.RipsComplex(points=data, max_edge_length=max_edge_length)
        st = rips.create_simplex_tree(max_dimension=2)
        diag = st.persistence()
        return st, diag

    st_c, diag_c = process_data(control_data)
    st_t, diag_t = process_data(treatment_data)

    # Helper for titles
    def set_title(ax, text):
        ax.set_title(text, fontweight='bold', fontsize=14)

    # --- ROW 1: CONTROL ---
    ax0 = fig.add_subplot(2, 3, 1, projection='3d')
    ax0.scatter(control_data[:, 0], control_data[:, 1], control_data[:, 2], s=10)
    set_title(ax0, f"Control Data (Edge: {max_edge_length:.5f})")

    ax1 = fig.add_subplot(2, 3, 2)
    gudhi.plot_persistence_diagram(diag_c, axes=ax1)
    set_title(ax1, "Control: Persistence")

    ax2 = fig.add_subplot(2, 3, 3)
    h1_c = st_c.persistence_intervals_in_dimension(1)
    if len(h1_c) > 0:
        LS = gudhi.representations.Landscape(num_landscapes=4, resolution=1000)
        lsc = LS.fit_transform([h1_c]).reshape(4, -1)
        for i in range(4): ax2.plot(lsc[i], label=f'Layer {i+1}')
    set_title(ax2, "Control: H1 Landscape")

    # --- ROW 2: TREATMENT ---
    ax3 = fig.add_subplot(2, 3, 4, projection='3d')
    ax3.scatter(treatment_data[:, 0], treatment_data[:, 1], treatment_data[:, 2], s=10, color='orange')
    set_title(ax3, "Treatment Data")

    ax4 = fig.add_subplot(2, 3, 5)
    gudhi.plot_persistence_diagram(diag_t, axes=ax4)
    set_title(ax4, "Treatment: Persistence")

    ax5 = fig.add_subplot(2, 3, 6)
    h1_t = st_t.persistence_intervals_in_dimension(1)
    if len(h1_t) > 0:
        LS = gudhi.representations.Landscape(num_landscapes=4, resolution=100)
        lsc = LS.fit_transform([h1_t]).reshape(4, -1)
        for i in range(4): ax5.plot(lsc[i], label=f'Layer {i+1}')
    set_title(ax5, "Treatment: H1 Landscape")

    plt.show()

# --- 3. Create the Interactive Slider ---
# We use SelectionSlider to strictly enforce 10 options from 0 to 1.5
steps = np.linspace(0, 2.0, 100)
edge_slider = widgets.SelectionSlider(
    options=[(f"{val:.5f}", val) for val in steps],
    value=0,
    description='Max Edge:',
    continuous_update=False, # Wait for mouse release to recalculate (prevents lag)
    layout={'width': '500px'}
)

# Link the slider to the function
interactive_ui = widgets.interactive(update_plots, max_edge_length=edge_slider)
display(interactive_ui)
